# Open AVIRIS-NG Mosaics from Bioscape
Shows how to access [BioSCape: AVIRIS-NG L3 Resampled Reflectance Mosaics, V2](https://doi.org/10.3334/ORNLDAAC/2427_

In [1]:
import earthaccess
import xarray as xr
import hvplot.xarray
import holoviews as hv
hvplot.extension('bokeh')

## Search for granules

In [2]:
auth = earthaccess.login()
granules = earthaccess.search_data(
    doi="10.3334/ORNLDAAC/2427", 
    temporal = ("2023-10-22", "2023-10-23")  ,
    bounding_box = (18.4, -34.3, 18.48, -34.18) ,
    granule_name=('*AVIRIS-NG_BIOSCAPE_V02_L3*')
)
print(f"Total granules found: {len(granules)}")

Total granules found: 4


## Print the first granule.

In [3]:
granules[0]

Collection: {'ShortName': 'BioSCape_ANG_V02_L3_RFL_Mosaic_2427', 'Version': '2'}
Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'GPolygons': [{'Boundary': {'Points': [{'Longitude': 18.4366, 'Latitude': -34.2271}, {'Longitude': 18.4428, 'Latitude': -34.1378}, {'Longitude': 18.3338, 'Latitude': -34.1327}, {'Longitude': 18.3275, 'Latitude': -34.222}, {'Longitude': 18.4366, 'Latitude': -34.2271}]}}]}}}
Temporal coverage: {'RangeDateTime': {'BeginningDateTime': '2023-10-22T00:00:00Z', 'EndingDateTime': '2023-11-26T23:59:59Z'}}
Size(MB): 7501.014312744141
Data: ['https://data.ornldaac.earthdata.nasa.gov/protected/bioscape/BioSCape_ANG_V02_L3_RFL_Mosaic/data/AVIRIS-NG_BIOSCAPE_V02_L3_37_11_QL.tif', 'https://data.ornldaac.earthdata.nasa.gov/protected/bioscape/BioSCape_ANG_V02_L3_RFL_Mosaic/data/AVIRIS-NG_BIOSCAPE_V02_L3_37_11_UNC.nc', 'https://data.ornldaac.earthdata.nasa.gov/protected/bioscape/BioSCape_ANG_V02_L3_RFL_Mosaic/data/AVIRIS-NG_BIOSCAPE_V02_L3_37_11_RFL.nc']

## Get Reflectance Granule S3 Links

In [4]:
def get_s3_links(g, suffix_str):
    return [i for i in g.data_links(access="direct") if i.endswith(suffix_str)][0]

rfl_f = []
for g in granules:
    rfl_f.append(get_s3_links(g, 'RFL.nc'))
rfl_f

['s3://ornl-cumulus-prod-protected/bioscape/BioSCape_ANG_V02_L3_RFL_Mosaic/data/AVIRIS-NG_BIOSCAPE_V02_L3_37_11_RFL.nc',
 's3://ornl-cumulus-prod-protected/bioscape/BioSCape_ANG_V02_L3_RFL_Mosaic/data/AVIRIS-NG_BIOSCAPE_V02_L3_37_12_RFL.nc',
 's3://ornl-cumulus-prod-protected/bioscape/BioSCape_ANG_V02_L3_RFL_Mosaic/data/AVIRIS-NG_BIOSCAPE_V02_L3_38_11_RFL.nc',
 's3://ornl-cumulus-prod-protected/bioscape/BioSCape_ANG_V02_L3_RFL_Mosaic/data/AVIRIS-NG_BIOSCAPE_V02_L3_38_12_RFL.nc']

## Open as S3FileSystem

In [5]:
s3_arr = earthaccess.open(rfl_f, provider='ornl_cloud')
s3_arr

QUEUEING TASKS | :   0%|          | 0/4 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/4 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/4 [00:00<?, ?it/s]

[<File-like object S3FileSystem, ornl-cumulus-prod-protected/bioscape/BioSCape_ANG_V02_L3_RFL_Mosaic/data/AVIRIS-NG_BIOSCAPE_V02_L3_37_11_RFL.nc>,
 <File-like object S3FileSystem, ornl-cumulus-prod-protected/bioscape/BioSCape_ANG_V02_L3_RFL_Mosaic/data/AVIRIS-NG_BIOSCAPE_V02_L3_37_12_RFL.nc>,
 <File-like object S3FileSystem, ornl-cumulus-prod-protected/bioscape/BioSCape_ANG_V02_L3_RFL_Mosaic/data/AVIRIS-NG_BIOSCAPE_V02_L3_38_11_RFL.nc>,
 <File-like object S3FileSystem, ornl-cumulus-prod-protected/bioscape/BioSCape_ANG_V02_L3_RFL_Mosaic/data/AVIRIS-NG_BIOSCAPE_V02_L3_38_12_RFL.nc>]

## Mosaic the tiles
Let's mosaic the first 4 tiles.

In [6]:
s3_obj = []
for fh in s3_arr[:4]:
    s3_obj.append(xr.open_datatree(fh, engine='h5netcdf', 
                                   chunks='auto').reflectance.to_dataset())
ds = xr.combine_by_coords(s3_obj, combine_attrs='override')

Notice the easting and northing dims below. One tile is 2000x2000. It has now increased to 6000x10000.

In [7]:
ds

<xarray.Dataset> Size: 54GB
Dimensions:      (easting: 4000, northing: 4000, wavelength: 425)
Coordinates:
  * easting      (easting) float64 32kB 7.9e+05 7.9e+05 ... 8.1e+05 8.1e+05
  * northing     (northing) float64 32kB 8.2e+05 8.2e+05 8.2e+05 ... 8e+05 8e+05
  * wavelength   (wavelength) float32 2kB 377.2 382.2 ... 2.496e+03 2.501e+03
Data variables:
    fwhm         (easting, northing, wavelength) float32 27GB dask.array<chunksize=(2000, 2000, 425), meta=np.ndarray>
    reflectance  (wavelength, northing, easting) float32 27GB dask.array<chunksize=(30, 768, 768), meta=np.ndarray>

## Plot True Color RGB image

In [8]:
ds_rgb = ds.reflectance.sel(wavelength=[637, 552, 462], method="nearest")

In [9]:
ds_rgb.hvplot.rgb('easting', 'northing', rasterize=True,robust=True, data_aspect=1, aspect='equal', 
                  bands='wavelength', frame_width=600)

:DynamicMap   []
   :RGB   [easting,northing]   (R,G,B)

## Plot a Spectra

In [10]:
ds.reflectance.sel(easting=798005,
                   northing=810005,
                   method="nearest").hvplot.line(x='wavelength', ylim=(0,0.5), color='green')

:Curve   [wavelength]   (Mosaiced Hemispherical Directional Reflectance Factor)